In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold

# 1. Завантаження даних COCO (зміни шлях, якщо датасет лежить в іншому місці)
dataset_dir = '../train' 
with open(os.path.join(dataset_dir, 'annotations_coco.json'), 'r') as f:
    coco_data = json.load(f)

images_df = pd.DataFrame(coco_data['images'])
anns_df = pd.DataFrame(coco_data['annotations'])

# Відфільтруємо ігноровані зони (iscrowd = 1) для аналізу
valid_anns_df = anns_df[anns_df['iscrowd'] == 0].copy()

# 2. EDA: Розподіл розмірів BBox
# COCO bbox формат: [x, y, w, h]
valid_anns_df[['x', 'y', 'w', 'h']] = pd.DataFrame(valid_anns_df['bbox'].tolist(), index=valid_anns_df.index)

plt.figure(figsize=(8, 6))
plt.scatter(valid_anns_df['w'], valid_anns_df['h'], alpha=0.1, s=1)
plt.title("Розподіл розмірів об'єктів (w vs h)")
plt.xlabel("Ширина (px)")
plt.ylabel("Висота (px)")
plt.axvline(x=30, color='r', linestyle='--', label='30 px')
plt.axhline(y=30, color='r', linestyle='--')
plt.legend()
plt.grid(True)
plt.show()

# 3. Валідація: Sequence-level split (GroupKFold)
# Витягуємо назву кліпу зі шляху (наприклад, images/clip_01/001.jpg -> clip_01)
images_df['clip'] = images_df['file_name'].apply(lambda x: x.split('/')[1])

gkf = GroupKFold(n_splits=5)
images_df['fold'] = -1

# Розбиваємо так, щоб усі кадри одного кліпу потрапляли строго в один фолд
for fold, (train_idx, val_idx) in enumerate(gkf.split(X=images_df, groups=images_df['clip'])):
    images_df.loc[val_idx, 'fold'] = fold

# Перевірка: кількість унікальних кліпів у кожному фолді
print("Кількість унікальних відео у кожному фолді:")
print(images_df.groupby('fold')['clip'].nunique())

# Зберігаємо розбиття для подальшого використання
images_df.to_csv('cv_splits.csv', index=False)

In [ ]:
import cv2
from pathlib import Path
from tqdm.notebook import tqdm

def mask_ignore_regions(base_dir, split_name):
    images_dir = Path(base_dir) / split_name / "images"
    ignore_dir = Path(base_dir) / split_name / "ignore"
    out_dir = Path(base_dir) / split_name / "images_masked"
    
    out_dir.mkdir(parents=True, exist_ok=True)
    clips = [d.name for d in images_dir.iterdir() if d.is_dir()]
    
    for clip in tqdm(clips, desc=f"Processing {split_name}"):
        (out_dir / clip).mkdir(parents=True, exist_ok=True)
        
        image_files = list((images_dir / clip).glob("*.jpg"))
        for img_path in image_files:
            frame_name = img_path.stem
            ignore_file = ignore_dir / clip / f"{frame_name}.txt"
            
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            h_img, w_img = img.shape[:2]
            
            # Якщо є файл з ігнорованими зонами, замальовуємо їх
            if ignore_file.exists():
                with open(ignore_file, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 4:
                            cx, cy, w, h = map(float, parts[:4])
                            # Перевід з нормалізованих координат у пікселі
                            x1 = int((cx - w / 2) * w_img)
                            y1 = int((cy - h / 2) * h_img)
                            x2 = int((cx + w / 2) * w_img)
                            y2 = int((cy + h / 2) * h_img)
                            
                            # Сірий колір (128, 128, 128)
                            cv2.rectangle(img, (x1, y1), (x2, y2), (128, 128, 128), -1)
            
            # Зберігаємо зображення (замасковане або оригінальне, якщо зон немає)
            out_path = out_dir / clip / f"{frame_name}.jpg"
            cv2.imwrite(str(out_path), img)

# Шлях до папки, де лежать train та val (оскільки ноутбук в notebooks/, піднімаємось на рівень вище)
dataset_root = "../" 

mask_ignore_regions(dataset_root, "train")
mask_ignore_regions(dataset_root, "val")
print("Маскування завершено!")

In [ ]:
import shutil
import os

# Копіюємо мітки для train
shutil.copytree('../train/labels', '../train/labels_masked', dirs_exist_ok=True)
# Копіюємо мітки для val
shutil.copytree('../val/labels', '../val/labels_masked', dirs_exist_ok=True)

print("Мітки успішно скопійовані!")